# 📖 Notebook 1: Introduction to Caching

Before we start implementing caching patterns, let's understand **why** caching exists, **where** you can cache, and **how much** it actually helps.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why reads are the bottleneck in most systems
- The difference between disk-based and memory-based storage
- The four places you can cache data
- How to measure the performance gap between database and cache

## 🛠️ Setup

Start the infrastructure first:

```bash
cd core-concepts/caching
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `caching_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "caching_demo",
    "user": "demo",
    "password": "demo"
}

# Redis connection settings
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True  # return strings instead of bytes
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test both connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker-compose up -d")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker-compose up -d")

## 🤔 Why Do We Need Caching?

Most applications have **far more reads than writes**. Think about it:

- You post 1 photo on Instagram, but thousands of people view it
- Amazon adds 1 product listing, but millions of shoppers browse it
- You shorten 1 URL on Bitly, but it gets clicked thousands of times

Every one of those reads hits your database. As traffic grows, the database becomes the bottleneck.

**The core idea of caching**: store frequently accessed data in fast memory (RAM) so you don't have to hit the slow database (disk) every time.

In [ ]:
# Let's see the read/write ratios in real-world systems

print("📊 Read/Write Ratios in Real Systems")
print("=" * 65)
print()

systems = [
    ("URL Shortener (Bitly)",   1000, 1, "1 URL created → 1000 clicks"),
    ("Social Media (Twitter)",   100, 1, "1 tweet → 100 views"),
    ("E-commerce (Amazon)",      500, 1, "1 product → 500 page views"),
    ("Video Platform (YouTube)",10000, 1, "1 upload → 10k views"),
    ("Banking App",                5, 1, "Check balance often, transact rarely"),
]

print(f"{'System':<26} {'Reads':>7} {'Writes':>7} {'Ratio':>8}  Notes")
print("-" * 80)

for system, reads, writes, notes in systems:
    ratio = f"{reads}:{writes}"
    bar = "█" * min(reads // 100, 20)
    print(f"{system:<26} {reads:>7} {writes:>7} {ratio:>8}  {notes}")

print()
print("💡 Optimizing reads has 10–1000× more impact than optimizing writes!")

## ⏱️ Disk vs Memory: The Speed Gap

Databases store data on **disk**. Every query pays the cost of disk I/O.  
Caches store data in **RAM**. RAM sits much closer to the CPU and avoids disk entirely.

Let's measure this difference ourselves with our e-commerce database.

In [ ]:
# Measure: How long does it take to read a product from PostgreSQL?

def fetch_product_from_db(product_id: int) -> dict:
    """Read a product from PostgreSQL (disk-based)."""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT id, name, price, rating_avg, view_count FROM products WHERE id = %s",
        (product_id,)
    )
    row = cursor.fetchone()
    conn.close()
    if row:
        return {"id": row[0], "name": row[1], "price": float(row[2]),
                "rating": float(row[3]), "views": row[4]}
    return None

# Run the query 100 times and measure average latency
times = []
for _ in range(100):
    start = time.time()
    product = fetch_product_from_db(42)
    elapsed = (time.time() - start) * 1000  # convert to milliseconds
    times.append(elapsed)

avg_db = sum(times) / len(times)
print(f"📦 Product from PostgreSQL (100 reads):")
print(f"   Average: {avg_db:.2f} ms")
print(f"   Min:     {min(times):.2f} ms")
print(f"   Max:     {max(times):.2f} ms")
print(f"   Data:    {product}")

In [ ]:
import json

# Now store that same product in Redis and measure read latency

r = get_redis_client()

# Write the product to Redis as a JSON string
r.set("product:42", json.dumps(product))

def fetch_product_from_cache(product_id: int) -> dict:
    """Read a product from Redis (memory-based)."""
    data = r.get(f"product:{product_id}")
    if data:
        return json.loads(data)
    return None

# Run the read 100 times and measure
times_cache = []
for _ in range(100):
    start = time.time()
    cached_product = fetch_product_from_cache(42)
    elapsed = (time.time() - start) * 1000
    times_cache.append(elapsed)

avg_cache = sum(times_cache) / len(times_cache)
print(f"⚡ Product from Redis (100 reads):")
print(f"   Average: {avg_cache:.2f} ms")
print(f"   Min:     {min(times_cache):.2f} ms")
print(f"   Max:     {max(times_cache):.2f} ms")
print()
print(f"🚀 Speedup: {avg_db / avg_cache:.1f}× faster with cache!")
print()
print("💡 This is why caching matters — memory is dramatically faster than disk.")

## 🗺️ Where to Cache

Caching isn't just Redis. There are **four layers** where you can cache data:

```
┌─────────────────────────────────────────────────────────┐
│  Layer 1: CLIENT-SIDE CACHE                             │
│  Browser cache, localStorage, mobile app local storage  │
│  ✅ No network call at all  ❌ Hard to invalidate       │
├─────────────────────────────────────────────────────────┤
│  Layer 2: CDN (Content Delivery Network)                │
│  Cloudflare, Akamai — caches static content at edge     │
│  ✅ Close to users globally  ❌ Mostly for static media │
├─────────────────────────────────────────────────────────┤
│  Layer 3: EXTERNAL CACHE (Redis / Memcached)            │
│  Shared cache all app servers talk to                   │
│  ✅ Shared, scalable, flexible  ❌ Network hop to Redis │
├─────────────────────────────────────────────────────────┤
│  Layer 4: IN-PROCESS CACHE                              │
│  Local memory inside each app server                    │
│  ✅ Fastest (no network)  ❌ Not shared across servers  │
└─────────────────────────────────────────────────────────┘
```

In system design interviews, **External Cache (Redis)** is the default answer.  
Mention the others only when the problem specifically calls for them.

In [ ]:
# Let's demonstrate in-process caching vs external (Redis) caching
# to see the difference even between cache layers

# In-process cache: just a Python dictionary in memory
local_cache = {}
local_cache["product:42"] = product

def fetch_product_from_local_cache(product_id: int) -> dict:
    """Read from in-process cache (Python dict in RAM)."""
    return local_cache.get(f"product:{product_id}")

# Measure in-process cache
times_local = []
for _ in range(100):
    start = time.time()
    _ = fetch_product_from_local_cache(42)
    elapsed = (time.time() - start) * 1000
    times_local.append(elapsed)

avg_local = sum(times_local) / len(times_local)

print("⚡ Latency Comparison (100 reads each):")
print("=" * 55)
print(f"  PostgreSQL (disk):     {avg_db:>8.3f} ms")
print(f"  Redis (network+RAM):   {avg_cache:>8.3f} ms")
print(f"  Python dict (local):   {avg_local:>8.3f} ms")
print()
print("💡 In-process is fastest, but it's not shared across servers.")
print("   Redis is the sweet spot: fast AND shared.")

## 📈 Why Caching Matters at Scale

Let's simulate what happens to our database under load — first without caching, then with it.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import random

def simulate_load(fetch_fn, label: str, num_requests: int = 200, workers: int = 20):
    """Simulate concurrent users hitting the system."""
    product_ids = [random.randint(1, 200) for _ in range(num_requests)]
    
    start = time.time()
    times = []
    
    def do_request(pid):
        t0 = time.time()
        fetch_fn(pid)
        return (time.time() - t0) * 1000
    
    with ThreadPoolExecutor(max_workers=workers) as pool:
        times = list(pool.map(do_request, product_ids))
    
    total = time.time() - start
    times.sort()
    
    print(f"📊 {label}")
    print(f"   Requests:    {num_requests}")
    print(f"   Concurrency: {workers} workers")
    print(f"   Total time:  {total:.2f}s")
    print(f"   Avg latency: {sum(times)/len(times):.2f} ms")
    print(f"   P95 latency: {times[int(len(times)*0.95)]:.2f} ms")
    print(f"   Throughput:  {num_requests/total:.0f} req/s")
    print()

# Without cache: every request goes to PostgreSQL
simulate_load(fetch_product_from_db, "WITHOUT CACHE (all reads hit PostgreSQL)")

# With cache: pre-populate Redis, then read from it
# First, warm the cache with all products
conn = get_db_connection()
cursor = conn.cursor()
cursor.execute("SELECT id, name, price, rating_avg, view_count FROM products")
for row in cursor.fetchall():
    product_data = {"id": row[0], "name": row[1], "price": float(row[2]),
                    "rating": float(row[3]), "views": row[4]}
    r.set(f"product:{row[0]}", json.dumps(product_data))
conn.close()

simulate_load(fetch_product_from_cache, "WITH CACHE (all reads hit Redis)")

print("💡 Notice how caching dramatically improves throughput and latency!")
print("   The database is freed up to handle writes and complex queries.")

## 🧭 The Four Cache Architectures (Preview)

How you read from and write to the cache changes performance, consistency, and complexity.  
There are **four core patterns** — we'll deep-dive into each in upcoming notebooks:

| Pattern | How It Works | Best For |
|---------|-------------|----------|
| **Cache-Aside** | App checks cache → miss → fetch DB → store in cache | Most common, default choice |
| **Write-Through** | App writes to cache → cache writes to DB synchronously | Reads must always be fresh |
| **Write-Behind** | App writes to cache → cache writes to DB asynchronously | High write throughput |
| **Read-Through** | Cache itself fetches from DB on miss | CDNs, specialized cache libraries |

**If you remember only one pattern for interviews, make it Cache-Aside.**

## 🧹 Cleanup

In [ ]:
# Clean up Redis keys we created
r = get_redis_client()
keys = r.keys("product:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")
else:
    print("🧹 Nothing to clean up")

## 📚 Summary

### Key Takeaways

1. **Reads dominate** — most apps are 10:1 to 1000:1 read/write ratio
2. **Memory is fast** — Redis is 10–50× faster than PostgreSQL for simple lookups
3. **Cache at the right layer** — External cache (Redis) is the default; mention others when relevant
4. **Caching adds complexity** — invalidation, consistency, and failure modes are real challenges

### Next Up

In **Notebook 2**, we'll implement the **Cache-Aside pattern** — the most common and most important caching pattern to know.